# Combined-feature leakage — a leak split across two columns

`LeakageCheck` asks whether a **single** column's solo predictive power
rivals the model's own. `CombinedFeatureLeakageCheck` asks the same
question of a **pair** of columns — because a leak does not have to live in
one column to be a leak.

This notebook builds a synthetic frame shaped exactly like the real case
that motivated the check: a public supply-chain dataset whose own
late-delivery label is defined by comparing two of its own columns, a
*scheduled* shipping window against the *realised* one. Each column alone
looks unremarkable. Together, they are the label.

For the real dataset, the real numbers, and a live demo of the finding, see
[chaqBD/bdp-gate-DataCo](https://github.com/chaqBD/bdp-gate-DataCo) (further
reading — the dataset is CC BY 4.0, via
[Mendeley Data](https://data.mendeley.com/datasets/8gx2fvg2k6)). Nothing
here depends on that repository or on downloading anything: the frame below
is generated, not fetched.

In [1]:
# %pip install -q "bdp-model-gate[structured]"
import numpy as np
import pandas as pd

from bdp_model_gate import StructuredGateContext
from bdp_model_gate.structured.validation_checks import (
    CombinedFeatureLeakageCheck,
    LeakageCheck,
)

rng = np.random.default_rng(7)

## 1. A label defined by comparing two columns

`scheduled` is the promised shipping window, set when the order is placed.
`real` is how long shipping actually took — known only once the order has
arrived. `late` is 1 exactly when `real` exceeds `scheduled`: the same
construction as the real dataset's `Late_delivery_risk`.

In [2]:
n = 20_000
# scheduled starts well clear of zero so real = scheduled + offset never
# goes negative — it is stored unsigned below, and a negative value would
# wrap to a large positive one rather than raise.
scheduled = rng.integers(10, 30, n)
offset = rng.choice([-1, 1], n) * rng.integers(1, 3, n)
real = scheduled + offset
late = (offset > 0).astype(int)

# A few ordinary, unrelated order features, so the frame looks like one a
# real gate would see rather than a two-column toy.
order_value = rng.lognormal(4.2, 0.6, n)
discount_rate = rng.uniform(0.0, 0.3, n)
quantity = rng.integers(1, 12, n)

X = pd.DataFrame(
    {
        "scheduled_days": scheduled.astype("uint16"),  # unsigned — the check covers this too
        "real_days": real.astype("uint16"),
        "order_value": order_value,
        "discount_rate": discount_rate,
        "quantity": quantity,
    }
)
y_true = late
y_pred = late.astype(float)  # a model that reproduces the label exactly
print(f"{n:,} rows · late-delivery rate {late.mean():.1%}")
X.head()

20,000 rows · late-delivery rate 50.5%


,scheduled_days,real_days,order_value,discount_rate,quantity
0,28,30,27.769057,0.081494,5
1,22,24,180.458097,0.004492,9
2,23,24,37.872038,0.149973,4
3,27,29,112.929495,0.179322,8
4,21,20,70.929973,0.158617,8


## 2. Neither column is suspicious alone

`LeakageCheck` measures each feature's solo predictive power on a 0–1 scale.
Both shipping-day columns land at roughly half its 0.80 flag threshold.

In [3]:
context = StructuredGateContext(model=None, X=X, y_true=y_true, y_pred=y_pred)
single = LeakageCheck().run(context)
for r in single:
    print(f"[{r.flag:>3}] {r.detail}")

[ OK] no single feature approaches the model's own predictive power (1.000)


## 3. The combination is the whole story

`CombinedFeatureLeakageCheck` fits a plain least-squares combination of
every candidate pair, scored out-of-fold, and compares it against the same
threshold. The pair the label was actually built from is exactly the one it
names.

In [4]:
combined = CombinedFeatureLeakageCheck().run(context)
for r in combined:
    print(f"[{r.flag}] blocking={r.blocking}")
    print(r.detail)
    print()
print("metadata:", {k: v for k, v in combined[0].metadata.items() if k != "individual_power"})

[COMBINED_LEAKAGE_RISK] blocking=False
scheduled_days + real_days together reach 1.000 of the target out-of-fold (100% of the whole model's 1.000), while no single one of them alone reaches 0.80 — check whether they are different views of the same post-outcome fact (e.g. an actual value and the plan or schedule it is compared against to define the label)

metadata: {'features': ['scheduled_days', 'real_days'], 'combo_power': 1.0, 'model_power': 1.0, 'power_ratio': 1.0, 'threshold_ratio': 0.95, 'min_power': 0.8, 'cv_folds': 5, 'n_candidates_scored': 5, 'n_combinations_tested': 10, 'suppressed_combinations': [], 'n_suppressed': 0}


Two things worth noting in the result above:

- **`blocking=False`.** Two columns matching the model's power is also the
  ordinary shape of a parsimonious, honest model — not only of a leak. The
  check names the pair for a reviewer rather than stopping the deploy on a
  finding that could, in principle, be a false positive.
- **`combo_power` vs. `model_power`** are nearly identical, because the
  model here learned nothing *but* the leak — see the next section for what
  that costs a benchmark when the model has other signal to fall back on
  too.

## 4. What this does to a benchmark

A model trained with `real_days` as a feature is not forecasting lateness —
it is checking the label's own arithmetic. Train the same model twice, once
with it and once without, and compare.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

feature_sets = {
    "naive (includes real_days)": ["scheduled_days", "real_days", "order_value", "discount_rate", "quantity"],
    "honest (excludes it)": ["scheduled_days", "order_value", "discount_rate", "quantity"],
}

results = {}
for label, features in feature_sets.items():
    X_train, X_val, y_train, y_val = train_test_split(
        X[features], y_true, test_size=0.25, random_state=0, stratify=y_true
    )
    model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=0, n_jobs=-1)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, proba)
    results[label] = auc
    print(f"{label:<28} ROC AUC {auc:.3f}")

inflation = results["naive (includes real_days)"] - results["honest (excludes it)"]
print(f"\nBenchmark inflation from including real_days: +{inflation:.3f} ROC AUC")

naive (includes real_days)   ROC AUC 0.999


honest (excludes it)         ROC AUC 0.502

Benchmark inflation from including real_days: +0.497 ROC AUC


Same synthetic dataset, same split, same model class. The gap is arithmetic,
not model skill — and it is exactly what `CombinedFeatureLeakageCheck`
named in section 3, before either model was ever trained.

## 5. Using it

```python
from bdp_model_gate.structured import default_structured_checks

checks = default_structured_checks()  # combined_feature_leakage is in here
```

No separate installation or wiring: it ships in core. See
[the reference docs](https://vanjy-eng.github.io/model-gate/docs/reference/checks/#combined_feature_leakage)
for the full set of `combined_leakage_*` config fields — combination size,
candidate cap, row-subsampling threshold, and whether columns `LeakageCheck`
already flags alone are excluded from the search.